# Demo 05: MPS comparison of the base model and fine-tuned adapter

Evaluate the original local Qwen3-1.7B and the Demo 01 LoRA adapter on the same held-out validation split. The scoring contract is identical to Demo 01: exact match, mean token F1, and accuracy at token F1 >= 0.80. This notebook requires MPS and performs the evaluations sequentially to reduce unified-memory pressure.

In [ ]:
import gc
import json
import platform
import re
import time
from collections import Counter
from pathlib import Path

import torch
from datasets import load_dataset
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer


def find_project_root(start_directory: Path) -> Path:
    """Return the repository root containing the project instructions."""
    for candidate in (start_directory, *start_directory.parents):
        if (candidate / 'AGENTS.md').is_file() and (candidate / 'requirements.txt').is_file():
            return candidate
    raise RuntimeError('Could not find the project root. Start JupyterLab from the repository root.')


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
MODEL_DIRECTORY = PROJECT_ROOT / 'artifacts' / 'models' / 'Qwen3-1.7B'
ADAPTER_DIRECTORY = PROJECT_ROOT / 'artifacts' / 'training' / 'demo01-qwen3-1.7b-lora' / 'adapter'
VALIDATION_FILE = PROJECT_ROOT / 'artifacts' / 'datasets' / 'cv-qa' / 'eval.jsonl'

for required_path in (MODEL_DIRECTORY / 'config.json', MODEL_DIRECTORY / 'tokenizer.json', VALIDATION_FILE, ADAPTER_DIRECTORY / 'adapter_config.json'):
    if not required_path.is_file():
        raise FileNotFoundError(f'Missing required local artifact: {required_path}')
if not any((ADAPTER_DIRECTORY / name).is_file() for name in ('adapter_model.safetensors', 'adapter_model.bin')):
    raise FileNotFoundError(f'Missing adapter weights in {ADAPTER_DIRECTORY}. Run Demo 01 first.')
single_weight_file = MODEL_DIRECTORY / 'model.safetensors'
weight_index_file = MODEL_DIRECTORY / 'model.safetensors.index.json'
if not single_weight_file.is_file() and not weight_index_file.is_file():
    raise FileNotFoundError(f'Missing model weights in {MODEL_DIRECTORY}. Expected model.safetensors or model.safetensors.index.json.')
if weight_index_file.is_file():
    shard_names = set(json.loads(weight_index_file.read_text(encoding='utf-8'))['weight_map'].values())
    missing_shards = sorted(name for name in shard_names if not (MODEL_DIRECTORY / name).is_file())
    if missing_shards:
        raise FileNotFoundError(f'Missing model weight shards in {MODEL_DIRECTORY}: {missing_shards}')

print(f'Base model: {MODEL_DIRECTORY}')
print(f'LoRA adapter: {ADAPTER_DIRECTORY}')
print(f'Validation split: {VALIDATION_FILE}')


In [ ]:
MODEL_DTYPE = torch.bfloat16
GENERATION_MAX_NEW_TOKENS = 128
TOKEN_F1_ACCURACY_THRESHOLD = 0.80

if not torch.backends.mps.is_built() or not torch.backends.mps.is_available():
    raise RuntimeError('This comparison requires an available MPS backend. Use a supported Apple Silicon PyTorch runtime; CPU fallback is intentionally disabled.')
macos_version = platform.mac_ver()[0]
macos_major = int(macos_version.split('.')[0]) if macos_version else 0
if macos_major < 14:
    raise RuntimeError(f'BF16 MPS comparison requires macOS 14 or later; detected {macos_version or "an unknown version"}.')

DEVICE = torch.device('mps')
print(f'PyTorch version: {torch.__version__}')
print(f'MPS built: {torch.backends.mps.is_built()}')
print(f'MPS available: {torch.backends.mps.is_available()}')
print(f'Selected device: {DEVICE}')
print(f'Model dtype: {MODEL_DTYPE}')
print(f'Maximum new tokens: {GENERATION_MAX_NEW_TOKENS}')
print(f'Token-F1 threshold: {TOKEN_F1_ACCURACY_THRESHOLD:.2f}')


In [ ]:
validation_dataset = load_dataset('json', data_files={'validation': str(VALIDATION_FILE)}, split='validation')
if len(validation_dataset) == 0:
    raise ValueError(f'Validation split is empty: {VALIDATION_FILE}')
for record in validation_dataset:
    messages = record.get('messages', [])
    if [message.get('role') for message in messages] != ['system', 'user', 'assistant']:
        raise ValueError(f'Invalid message roles in validation record {record.get("id", "<unknown>")}.')
    if any(not message.get('content', '').strip() for message in messages):
        raise ValueError(f'Empty message content in validation record {record.get("id", "<unknown>")}.')
print(f'Validated records: {len(validation_dataset)}')


In [ ]:
def normalise_for_scoring(text: str) -> list[str]:
    """Return lowercase word tokens for the Demo 01 factual-overlap metric."""
    return re.findall(r'\w+', text.casefold())


def token_f1(prediction: str, reference: str) -> float:
    """Calculate the same bag-of-token F1 used by Demo 01."""
    prediction_counts = Counter(normalise_for_scoring(prediction))
    reference_counts = Counter(normalise_for_scoring(reference))
    if not prediction_counts or not reference_counts:
        return 0.0
    overlap = sum((prediction_counts & reference_counts).values())
    precision = overlap / sum(prediction_counts.values())
    recall = overlap / sum(reference_counts.values())
    return 2 * precision * recall / (precision + recall) if precision + recall else 0.0


def evaluate_model(model, tokenizer, label: str) -> dict[str, float]:
    """Generate deterministic answers and return Demo 01 aggregate metrics."""
    model.eval()
    scores = []
    started_at = time.perf_counter()
    for record in validation_dataset:
        prompt_text = tokenizer.apply_chat_template(record['messages'][:-1], tokenize=False, add_generation_prompt=True, enable_thinking=False)
        model_inputs = {name: tensor.to(DEVICE) for name, tensor in tokenizer(prompt_text, return_tensors='pt').items()}
        with torch.inference_mode():
            output_ids = model.generate(**model_inputs, max_new_tokens=GENERATION_MAX_NEW_TOKENS, do_sample=False, pad_token_id=tokenizer.eos_token_id)
        prediction = tokenizer.decode(output_ids[0, model_inputs['input_ids'].shape[-1]:], skip_special_tokens=True).strip()
        reference = record['messages'][-1]['content']
        f1_score = token_f1(prediction, reference)
        scores.append((prediction.casefold().strip() == reference.casefold().strip(), f1_score))
    elapsed_seconds = time.perf_counter() - started_at
    return {
        'exact_match_rate': sum(exact_match for exact_match, _ in scores) / len(scores),
        'mean_token_f1': sum(f1_score for _, f1_score in scores) / len(scores),
        'threshold_accuracy': sum(f1_score >= TOKEN_F1_ACCURACY_THRESHOLD for _, f1_score in scores) / len(scores),
        'elapsed_seconds': elapsed_seconds,
    }


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIRECTORY, local_files_only=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(MODEL_DIRECTORY, dtype=MODEL_DTYPE, local_files_only=True).to(DEVICE)
if base_model.config.model_type != 'qwen3':
    raise RuntimeError(f'Expected a Qwen3 base model, got {base_model.config.model_type!r}.')
if next(base_model.parameters()).device.type != 'mps' or next(base_model.parameters()).dtype != MODEL_DTYPE:
    raise RuntimeError('Base model was not loaded on MPS in BF16.')

base_metrics = evaluate_model(base_model, tokenizer, 'Base model')
del base_model
gc.collect()
torch.mps.empty_cache()
print('Base-model evaluation completed; model memory released.')


In [ ]:
adapter_base_model = AutoModelForCausalLM.from_pretrained(MODEL_DIRECTORY, dtype=MODEL_DTYPE, local_files_only=True).to(DEVICE)
fine_tuned_model = PeftModel.from_pretrained(adapter_base_model, ADAPTER_DIRECTORY, local_files_only=True, autocast_adapter_dtype=False).to(DEVICE)
if not isinstance(fine_tuned_model, PeftModel):
    raise RuntimeError('The fine-tuned model is not a PEFT model.')
if next(fine_tuned_model.parameters()).device.type != 'mps' or next(fine_tuned_model.parameters()).dtype != MODEL_DTYPE:
    raise RuntimeError('Fine-tuned model was not loaded on MPS in BF16.')

fine_tuned_metrics = evaluate_model(fine_tuned_model, tokenizer, 'Fine-tuned adapter')
print('Fine-tuned-adapter evaluation completed.')


In [ ]:
comparison_rows = [
    ('Exact-match rate', 'exact_match_rate', '.1%'),
    ('Mean token F1', 'mean_token_f1', '.3f'),
    (f'Accuracy at token-F1 >= {TOKEN_F1_ACCURACY_THRESHOLD:.2f}', 'threshold_accuracy', '.1%'),
]
print('| Metric | Base model | Fine-tuned adapter | Delta (fine-tuned - base) |')
print('| --- | ---: | ---: | ---: |')
for display_name, metric_name, score_format in comparison_rows:
    base_score = base_metrics[metric_name]
    fine_tuned_score = fine_tuned_metrics[metric_name]
    delta = fine_tuned_score - base_score
    if score_format == '.1%':
        delta_display = f'{delta:+.1%}'
    else:
        delta_display = f'{delta:+.3f}'
    print(f'| {display_name} | {base_score:{score_format}} | {fine_tuned_score:{score_format}} | {delta_display} |')
print(f"\nElapsed evaluation time (base): {base_metrics['elapsed_seconds']:.1f} seconds")
print(f"Elapsed evaluation time (fine-tuned): {fine_tuned_metrics['elapsed_seconds']:.1f} seconds")
